# Theorem 2 workflow: environment-conditioned response modes

This notebook keeps **Task A** unchanged and redesigns **Task B** as an environment-comparison experiment.

The same underlying arithmetic problems are used across Task B environments, but each environment teaches a different response policy:

- `artifact_only`: `P -> [ANS] S`
- `worked_trace`: `P -> [WORK] derivation -> [ANS] S`
- `failed_then_repair`: `P -> [TRY] wrong attempt -> [FAIL] [REPAIR] derivation -> [ANS] S`

We then evaluate the **same learner class** in two deployment modes:

- **direct-answer mode**: emit the final answer directly
- **process mode**: emit a process trace and eventually the answer

This supports the two-sided Theorem 2 interpretation:

- artefact environments can help direct artefact emission
- process-rich environments can help process-sensitive deployment


In [ ]:

from pathlib import Path
import sys
import json
import pandas as pd

# Auto-discover project root
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if p.name == "Drift_and_selection":
            return p
        if (p / "GitHub").exists() and (p / "Nat_Paper").exists():
            return p
    return start

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "GitHub" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    from drift_selection.theorem2_process_learning import (
        TaskConfig, ModelConfig, TrainConfig,
        build_task_family_dataset, pretty_print_examples,
        run_task_pipeline, default_output_roots
    )
except Exception:
    # fallback for running from the exported artifact location
    helper_local = Path.cwd() / "theorem2_process_learning.py"
    if helper_local.exists():
        sys.path.insert(0, str(Path.cwd()))
        from theorem2_process_learning import (
            TaskConfig, ModelConfig, TrainConfig,
            build_task_family_dataset, pretty_print_examples,
            run_task_pipeline, default_output_roots
        )
    else:
        raise

ROOTS = default_output_roots(PROJECT_ROOT)
ROOTS


## Task design

### Task family A — column addition (unchanged)
Prompt:
- two integers as digit tokens

Target styles:
- `process`: full column-by-column trace
- `artifact`: final answer only

Expected interpretation:
- if the task is process-sensitive, process traces should improve downstream success.

### Task family B — same task, different visible training environments
All Task B styles use the **same underlying sampled arithmetic problems**.
Only the visible trace structure changes.

| Training environment | Visible target structure | Best-matched deployment mode |
| --- | --- | --- |
| `artifact_only` | final answer artefact only | direct-answer mode |
| `worked_trace` | successful derivation then answer | process mode |
| `failed_then_repair` | failed attempt + explicit repair + answer | repair-aware process mode |

This setup isolates an environment effect (what traces enter the public corpus), not an architecture effect.


In [ ]:
# ------------------------------------------------------------------
# Global run toggles + pilot profiles
# ------------------------------------------------------------------

RUN_BUILD_PREVIEWS = True
RUN_TASK_A = True
RUN_TASK_B = True
RUN_FULL_WORKFLOW = False   # set True only when you want everything

TASK_B_VARIANT = "environment_modes_v2"

# Profiles:
# - pilot_in_dist: train/test both 2-3 digits (recommended first)
# - pilot_mild_ood: train 2-3 digits, test 4 digits
# - hard_ood: train 2-3 digits, test 4-5 digits
RUN_PROFILE = "pilot_in_dist"

PROFILE_PRESETS = {
    "pilot_in_dist": {
        "a_train": (2, 3), "a_test": (2, 3),
        "b_train": (2, 3), "b_test": (2, 3),
        "epochs": 8,
        "a_sizes": (8000, 1000, 2000),
        "b_sizes": (9000, 1200, 2000),
    },
    "pilot_mild_ood": {
        "a_train": (2, 3), "a_test": (4, 4),
        "b_train": (2, 3), "b_test": (4, 4),
        "epochs": 8,
        "a_sizes": (9000, 1200, 2000),
        "b_sizes": (9000, 1200, 2000),
    },
    "hard_ood": {
        "a_train": (2, 3), "a_test": (4, 5),
        "b_train": (2, 3), "b_test": (4, 5),
        "epochs": 10,
        "a_sizes": (12000, 1500, 2500),
        "b_sizes": (12000, 1500, 2500),
    },
}

if RUN_PROFILE not in PROFILE_PRESETS:
    raise ValueError(f"Unknown RUN_PROFILE: {RUN_PROFILE}")
P = PROFILE_PRESETS[RUN_PROFILE]

TASK_CFG_A = TaskConfig(
    train_min_digits=P["a_train"][0],
    train_max_digits=P["a_train"][1],
    test_min_digits=P["a_test"][0],
    test_max_digits=P["a_test"][1],
    n_train=P["a_sizes"][0],
    n_val=P["a_sizes"][1],
    n_test=P["a_sizes"][2],
    seed=123,
)

TASK_CFG_B = TaskConfig(
    train_min_digits=P["b_train"][0],
    train_max_digits=P["b_train"][1],
    test_min_digits=P["b_test"][0],
    test_max_digits=P["b_test"][1],
    n_train=P["b_sizes"][0],
    n_val=P["b_sizes"][1],
    n_test=P["b_sizes"][2],
    seed=321,
)

MODEL_CFG = ModelConfig(
    d_model=128,
    n_heads=4,
    n_layers=2,
    d_ff=512,
    dropout=0.1,
    max_seq_len=256,
)

TRAIN_CFG = TrainConfig(
    batch_size=64,
    learning_rate=3e-4,
    weight_decay=0.01,
    epochs=P["epochs"],
    clip_grad_norm=1.0,
    eval_every_epoch=True,
    seed=123,
    checkpoint_every_epoch=True,
)

TASK_B_STYLES = ["artifact_only", "worked_trace", "failed_then_repair"]

RUN_SUFFIX = f"{RUN_PROFILE}_ep{TRAIN_CFG.epochs}"
TASK_A_RUN_NAME = f"theorem2_taskA_addition_process_vs_artifact__{RUN_SUFFIX}"
TASK_B_RUN_NAME = "taskB_environment_modes_v2"

ROOTS = default_output_roots(PROJECT_ROOT)
print("RUN_PROFILE:", RUN_PROFILE)
print("Task A run:", TASK_A_RUN_NAME)
print("Task B run:", TASK_B_RUN_NAME)
ROOTS


In [ ]:
# ------------------------------------------------------------------
# Preview generated examples before training
# ------------------------------------------------------------------

if RUN_BUILD_PREVIEWS:
    ds_a = build_task_family_dataset(
        "addition_process_vs_artifact",
        styles=["process", "artifact"],
        cfg=TASK_CFG_A,
    )
    print("Task A — process examples")
    print(pretty_print_examples(ds_a["process"]["train"], n=3))
    print("Task A — artifact examples")
    print(pretty_print_examples(ds_a["artifact"]["train"], n=3))

    ds_b = build_task_family_dataset(
        "taskB_environment_modes_v2",
        styles=TASK_B_STYLES,
        cfg=TASK_CFG_B,
    )
    print("Task B — artifact_only examples")
    print(pretty_print_examples(ds_b["artifact_only"]["train"], n=2))
    print("Task B — worked_trace examples")
    print(pretty_print_examples(ds_b["worked_trace"]["train"], n=2))
    print("Task B — failed_then_repair examples")
    print(pretty_print_examples(ds_b["failed_then_repair"]["train"], n=2))


## Pilot run recommendations

Run in this order:

1. **Task B direct/process comparison (in-distribution first)**
2. **Task B mild OOD** (`RUN_PROFILE = "pilot_mild_ood"`)
3. **Task A** as the negative-Theorem-2 baseline

For Task B, the key output files are split by deployment mode:

- `evaluation_direct/summary_metrics.csv`
- `evaluation_process/summary_metrics.csv`
- `figures/figure_B1_direct_answer_metrics.pdf`
- `figures/figure_B2_process_mode_metrics.pdf`
- `figures/figure_B3_direct_mode_marker_contamination.pdf`


In [ ]:
# ------------------------------------------------------------------
# Task A: full worked addition traces vs final-answer-only
# ------------------------------------------------------------------

task_a_result = None

if RUN_TASK_A or RUN_FULL_WORKFLOW:
    task_a_result = run_task_pipeline(
        task_family="addition_process_vs_artifact",
        styles=["process", "artifact"],
        task_cfg=TASK_CFG_A,
        model_cfg=MODEL_CFG,
        train_cfg=TRAIN_CFG,
        run_name=TASK_A_RUN_NAME,
        project_root=PROJECT_ROOT,
    )
    display(task_a_result["metrics_df"])
    print("Task A outputs:", task_a_result["run_root"])


In [ ]:
# ------------------------------------------------------------------
# Task B redesign: same task, different training environments
# ------------------------------------------------------------------

task_b_result = None

if RUN_TASK_B or RUN_FULL_WORKFLOW:
    task_b_result = run_task_pipeline(
        task_family="taskB_environment_modes_v2",
        styles=TASK_B_STYLES,
        task_cfg=TASK_CFG_B,
        model_cfg=MODEL_CFG,
        train_cfg=TRAIN_CFG,
        run_name=TASK_B_RUN_NAME,
        project_root=PROJECT_ROOT,
    )
    print("Task B outputs:", task_b_result["run_root"])
    print("Direct mode metrics")
    display(task_b_result.get("direct_metrics_df", task_b_result["metrics_df"]))
    print("Process mode metrics")
    display(task_b_result.get("process_metrics_df", task_b_result["metrics_df"]))


## Reading the outputs

### Task A (unchanged)
Primary metric:
- `answer_accuracy`

### Task B (environment-modes redesign)
Direct-answer mode metrics:
- `direct_exact_match`
- `direct_clean_answer_rate`
- `direct_marker_contamination_rate`
- `direct_answer_logprob`

Process mode metrics:
- `process_final_answer_accuracy`
- `process_trace_validity`
- `repair_marker_accuracy`
- `process_answer_in_completion`

Interpretation target:
- **direct mode**: `artifact_only` should usually be strongest/cleanest
- **process mode**: `worked_trace` and/or `failed_then_repair` should usually improve process-sensitive behaviour


In [ ]:
# ------------------------------------------------------------------
# Optional: inspect saved metrics after runs complete
# ------------------------------------------------------------------

def load_task_a_summary_if_exists(run_name: str):
    csv_path = ROOTS["data_out"] / run_name / "evaluation" / "summary_metrics.csv"
    if csv_path.exists():
        print("Task A:", run_name)
        display(pd.read_csv(csv_path))
    else:
        print(f"No Task A summary found for {run_name}")


def load_task_b_summaries_if_exists(run_name: str):
    run_root = ROOTS["data_out"] / run_name
    direct_csv = run_root / "evaluation_direct" / "summary_metrics.csv"
    process_csv = run_root / "evaluation_process" / "summary_metrics.csv"
    if direct_csv.exists():
        print("Task B direct mode:", run_name)
        display(pd.read_csv(direct_csv))
    else:
        print(f"No direct-mode summary found for {run_name}")
    if process_csv.exists():
        print("Task B process mode:", run_name)
        display(pd.read_csv(process_csv))
    else:
        print(f"No process-mode summary found for {run_name}")


load_task_a_summary_if_exists(TASK_A_RUN_NAME)
load_task_b_summaries_if_exists(TASK_B_RUN_NAME)


## Next extensions

- add mild OOD and hard OOD comparisons for Task B
- contrast direct-mode contamination rates against answer accuracy
- reuse this setup to populate Theorem 2 appendix panels (negative and positive sides)
